# IIC-3641 GML UC

- Versiones de librerías, python 3.10.2
- DGL: https://www.dgl.ai/pages/start.html


In [1]:
import torch
print(torch.__version__)

2.7.1+cu118


## DGL requiere de un framework de backend. Aquí va con torch sobre cuda (GPU).

In [2]:
import os

os.environ["DGLBACKEND"] = "pytorch"
import dgl
import dgl.function as fn
import torch as th
import torch.nn as nn
import torch.nn.functional as F
from dgl import DGLGraph
import time
import numpy as np

gcn_msg = fn.copy_u(u="h", out="m")
gcn_reduce = fn.sum(msg="m", out="h")

## Leemos el dataset

In [3]:
from dgl.data import CoraGraphDataset


def load_cora_data():
    dataset = CoraGraphDataset()
    g = dataset[0]
    features = g.ndata["feat"]
    labels = g.ndata["label"]
    train_mask = g.ndata["train_mask"]
    test_mask = g.ndata["test_mask"]
    return g, features, labels, train_mask, test_mask

In [4]:
g, features, labels, train_mask, test_mask = load_cora_data()
# Add edges between each node and itself to preserve old node representations
g.add_edges(g.nodes(), g.nodes())

  NumNodes: 2708
  NumEdges: 10556
  NumFeats: 1433
  NumClasses: 7
  NumTrainingSamples: 140
  NumValidationSamples: 500
  NumTestSamples: 1000
Done loading data from cached files.


## Definimos una capa GCN

In [5]:
class GCNLayer(nn.Module):
    def __init__(self, in_feats, out_feats):
        super(GCNLayer, self).__init__()
        self.linear = nn.Linear(in_feats, out_feats)

    def forward(self, g, feature):
        with g.local_scope():
            g.ndata["h"] = feature
            g.update_all(gcn_msg, gcn_reduce)
            h = g.ndata["h"]
            return self.linear(h)

## Y definimos la red, en este caso de dos capas

In [6]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.layer1 = GCNLayer(1433, 16)
        self.layer2 = GCNLayer(16, 7)

    def forward(self, g, features):
        x = F.relu(self.layer1(g, features))
        x = self.layer2(g, x)
        return x


net = Net()
print(net)


Net(
  (layer1): GCNLayer(
    (linear): Linear(in_features=1433, out_features=16, bias=True)
  )
  (layer2): GCNLayer(
    (linear): Linear(in_features=16, out_features=7, bias=True)
  )
)


In [7]:
def evaluate(model, g, features, labels, mask):
    model.eval()
    with th.no_grad():
        logits = model(g, features)
        logits = logits[mask]
        labels = labels[mask]
        _, indices = th.max(logits, dim=1)
        correct = th.sum(indices == labels)
        return correct.item() * 1.0 / len(labels)

## Y entrenamos

In [8]:
optimizer = th.optim.Adam(net.parameters(), lr=1e-2)
dur = []
for epoch in range(100):
    if epoch >= 0:
        t0 = time.time()
    net.train()
    logits = net(g, features)
    logp = F.log_softmax(logits, 1)
    loss = F.nll_loss(logp[train_mask], labels[train_mask])

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch >= 0:
        dur.append(time.time() - t0)
    acc = evaluate(net, g, features, labels, test_mask)
    print(
        "Epoch {:05d} | Loss {:.4f} | Test Acc {:.4f} | Time(s) {:.4f}".format(
            epoch, loss.item(), acc, np.mean(dur)
        )
    )

Epoch 00000 | Loss 1.9419 | Test Acc 0.2480 | Time(s) 0.0709
Epoch 00001 | Loss 1.7872 | Test Acc 0.5190 | Time(s) 0.0384
Epoch 00002 | Loss 1.6561 | Test Acc 0.5770 | Time(s) 0.0272
Epoch 00003 | Loss 1.5578 | Test Acc 0.6030 | Time(s) 0.0216
Epoch 00004 | Loss 1.4709 | Test Acc 0.6180 | Time(s) 0.0182
Epoch 00005 | Loss 1.3843 | Test Acc 0.6570 | Time(s) 0.0160
Epoch 00006 | Loss 1.2968 | Test Acc 0.6910 | Time(s) 0.0144
Epoch 00007 | Loss 1.2082 | Test Acc 0.7140 | Time(s) 0.0132
Epoch 00008 | Loss 1.1192 | Test Acc 0.7220 | Time(s) 0.0123
Epoch 00009 | Loss 1.0322 | Test Acc 0.7370 | Time(s) 0.0115
Epoch 00010 | Loss 0.9508 | Test Acc 0.7390 | Time(s) 0.0109
Epoch 00011 | Loss 0.8735 | Test Acc 0.7290 | Time(s) 0.0104
Epoch 00012 | Loss 0.7977 | Test Acc 0.7210 | Time(s) 0.0100
Epoch 00013 | Loss 0.7292 | Test Acc 0.7210 | Time(s) 0.0096
Epoch 00014 | Loss 0.6680 | Test Acc 0.7240 | Time(s) 0.0093
Epoch 00015 | Loss 0.6099 | Test Acc 0.7240 | Time(s) 0.0090
Epoch 00016 | Loss 0.554

In [9]:
import warnings

# seaborn 0.12 usa una opción de pandas ya deprecada en pandas 2.2; el aviso no aporta nada acá
warnings.filterwarnings("ignore", message="use_inf_as_na option is deprecated")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Cora no guarda los nombres de las clases, solo índices 0..6. El mapeo se
# verifica por el tamaño de cada clase: los 7 tamaños son distintos, así que
# la asignación es única.
CLASSES = [
    "Theory",                  # 0 -> 351 nodos
    "Reinforcement_Learning",  # 1 -> 217
    "Genetic_Algorithms",      # 2 -> 418
    "Neural_Networks",         # 3 -> 818
    "Probabilistic_Methods",   # 4 -> 426
    "Case_Based",              # 5 -> 298
    "Rule_Learning",           # 6 -> 180
]

val_mask = g.ndata["val_mask"]   # el dataset trae split de validación; el loop de arriba no lo usa

net.eval()
with th.no_grad():
    logits = net(g, features)
    probs = F.softmax(logits, dim=1)
    preds = logits.argmax(dim=1)

y_true = labels[test_mask].numpy()
y_pred = preds[test_mask].numpy()

for nombre, m in [("train", train_mask), ("val", val_mask), ("test", test_mask)]:
    acc = (preds[m] == labels[m]).float().mean().item()
    print(f"Accuracy {nombre:>5}: {acc:.4f}   ({int(m.sum())} nodos)")

Accuracy train: 1.0000   (140 nodos)
Accuracy   val: 0.7680   (500 nodos)
Accuracy  test: 0.7490   (1000 nodos)


In [10]:
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3, zero_division=0))

rep = classification_report(y_true, y_pred, target_names=CLASSES,
                            output_dict=True, zero_division=0)
por_clase = (pd.DataFrame(rep).T.loc[CLASSES]
             .assign(support=lambda d: d["support"].astype(int))
             .sort_values("f1-score", ascending=False))

(por_clase.style
 .background_gradient(cmap="RdYlGn", subset=["precision", "recall", "f1-score"], vmin=0, vmax=1)
 .format({"precision": "{:.3f}", "recall": "{:.3f}", "f1-score": "{:.3f}"})
 .set_caption("Métricas por clase en el conjunto de test"))

                        precision    recall  f1-score   support

                Theory      0.633     0.769     0.694       130
Reinforcement_Learning      0.661     0.835     0.738        91
    Genetic_Algorithms      0.777     0.799     0.788       144
       Neural_Networks      0.893     0.680     0.772       319
 Probabilistic_Methods      0.723     0.752     0.737       149
            Case_Based      0.786     0.748     0.766       103
         Rule_Learning      0.627     0.812     0.707        64

              accuracy                          0.749      1000
             macro avg      0.728     0.771     0.743      1000
          weighted avg      0.768     0.749     0.751      1000



,precision,recall,f1-score,support
Genetic_Algorithms,0.777,0.799,0.788,144
Neural_Networks,0.893,0.680,0.772,319
Case_Based,0.786,0.748,0.766,103
Reinforcement_Learning,0.661,0.835,0.738,91
Probabilistic_Methods,0.723,0.752,0.737,149
Rule_Learning,0.627,0.812,0.707,64
Theory,0.633,0.769,0.694,130


In [12]:
u, v = g.edges()
sin_self = u != v
u, v = u[sin_self], v[sin_self]
mismo = (labels[u] == labels[v]).float()

n = g.num_nodes()
vecinos_iguales = th.zeros(n).index_add_(0, v, mismo)
grado = th.zeros(n).index_add_(0, v, th.ones_like(mismo))
homofilia = vecinos_iguales / grado.clamp(min=1)

resumen = pd.DataFrame({
    "clase":       CLASSES,
    "n_total":     [int((labels == c).sum()) for c in range(len(CLASSES))],
    "n_train":     [int((labels[train_mask] == c).sum()) for c in range(len(CLASSES))],
    "n_test":      [int((labels[test_mask] == c).sum()) for c in range(len(CLASSES))],
    "predichos":   [int((preds[test_mask] == c).sum()) for c in range(len(CLASSES))],
    "grado medio": [grado[labels == c].mean().item() for c in range(len(CLASSES))],
    "homofilia":   [homofilia[labels == c].mean().item() for c in range(len(CLASSES))],
    "recall":      [por_clase.loc[CLASSES[c], "recall"] for c in range(len(CLASSES))],
    "precision":   [por_clase.loc[CLASSES[c], "precision"] for c in range(len(CLASSES))],
}).sort_values("recall", ascending=False)

print("Homofilia global del grafo: %.3f" % homofilia.mean().item())
for col in ["grado medio", "homofilia"]:
    r = resumen[col].corr(resumen["recall"])
    print(f"correlación  {col:>12} vs recall: {r:+.3f}")
print()
resumen.round(3)

Homofilia global del grafo: 0.825
correlación   grado medio vs recall: +0.702
correlación     homofilia vs recall: -0.178



,clase,n_total,n_train,n_test,predichos,grado medio,homofilia,recall,precision
1,Reinforcement_Learning,217,20,91,115,4.742,0.769,0.835,0.661
6,Rule_Learning,180,20,64,83,3.656,0.788,0.812,0.627
2,Genetic_Algorithms,418,20,144,148,4.368,0.917,0.799,0.777
0,Theory,351,20,130,158,4.350,0.743,0.769,0.633
4,Probabilistic_Methods,426,20,149,155,3.737,0.849,0.752,0.723
5,Case_Based,298,20,103,98,3.644,0.786,0.748,0.786
3,Neural_Networks,818,20,319,243,3.469,0.839,0.680,0.893
